## S8: Export the final dataset

Last step of the patents pipeline. Reads the finished table `patents_labelled` and writes it to a CSV in `data/`.

`patents_labelled` is built by `S7_LLM_labelling.py` and contains only patents that made it through the whole chain: retrieved from Dimensions (S1) → ML pre-filter (S2) → LLM scope check (S3) → family members added (S4) → assignees cleaned (S5) → reviewed (S6) → all four labels assigned (S7). Working columns (probabilities, LLM diagnostics) are already stripped out; what remains is patent metadata plus `scope_curated`, `pillar_curated`, `research_category`, `end_product`, `ingredient` and `subpillar`.

Nothing is changed here - the notebook only reads and writes a file, so it can be re-run at any time. Note that patents missing even one of the four labels are not in this table; they stay in `patents_classified` and are retried on the next S7 run.

In [7]:
import duckdb
import pandas as pd
from datetime import datetime

# today
# goes into the export file name, so every export is dated and older ones are not overwritten
today = datetime.today().strftime("%y%m%d")

In [8]:
# Connect to database
# read-only use here, so no PRAGMA settings are needed (unlike S6_Review_classifications.ipynb)
db = duckdb.connect('patents.db')

In [9]:
# Show tables in database
# just to check that patents_labelled exists and to see which runs are in the database
db.sql("SHOW TABLES")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ patents_classified      │
│ patents_embeddings      │
│ patents_labelled        │
│ run_260710_1614         │
│ run_260710_1614_reverse │
│ run_260713_0858         │
│ run_260713_0858_reverse │
│ run_260724_1438         │
│ run_260724_1438_reverse │
└─────────────────────────┘

In [10]:
# get labelled data
# worth a look before exporting: df.shape and e.g. df['pillar_curated'].value_counts() show whether
# the export is complete and plausible.
# Note that nested fields (assignee_cities, assignee_countries, publications) are stored as JSON text
# and need parsing during analysis; assignee_names_cleaned, on the other hand, is a real list.
df = db.sql("SELECT * FROM patents_labelled").df()

In [11]:
# close the connection so the database file is not left locked (the data is already in df, so the
# export in the next cell works without a connection)
db.close()

In [ ]:
# save data as csv
# lands in data/ with today's date. index=False avoids an extra unnamed index column, and
# encoding='utf-8-sig' makes Excel on Windows display special characters correctly.
df.to_csv(f"data/Patents_data_labelled_{today}.csv", index=False, encoding='utf-8-sig')